# Primeiros passos no Google Earth Engine com Python

Leia antes as aulas 1 a 4 em [Google Earth Engine com Python](../README.md).

Neste notebook, você vai:

1. Autenticar e inicializar a sessão do GEE com o seu `project ID` do Google Cloud.
2. Conferir a diferença entre um objeto do servidor (`ee.*`) e um valor local em Python.
3. Filtrar imagens `COPERNICUS/S2_SR_HARMONIZED` na região de Cuiabá (MT).
4. Aplicar fator de escala para reflectância de superfície e máscara de nuvens (`SCL` e opção `Cloud Score+`).
5. Montar uma composição mediana, calcular NDVI e NDWI (McFeeters) e visualizar em mapa interativo com `geemap`.

Pré-requisito: ambiente com `earthengine-api` e `geemap` instalados (`conteudo/google-earth-engine/requirements-gee.txt`) e conta vinculada a um projeto no Google Cloud registrado no Earth Engine. Os exemplos referenciam dados Sentinel-2 (Contains modified Copernicus Sentinel data 2024) e SRTM (USGS/NASA).


## 1. Autenticação e inicialização

Defina a variável de ambiente `GEE_PROJECT_ID` antes de abrir o Jupyter (por exemplo, `export GEE_PROJECT_ID="seu-projeto"`) ou substitua `"SEU_PROJECT_ID_AQUI"` localmente sem commitar o seu ID pessoal. Na primeira vez nesta máquina (ou em uma nova sessão do Google Colab), descomente `ee.Authenticate()`.


In [ ]:
import os
import ee
import geemap

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "SEU_PROJECT_ID_AQUI")

# Na primeira execução neste computador (ou no Colab), descomente a linha abaixo:
# ee.Authenticate()

ee.Initialize(project=GEE_PROJECT_ID)
print("Sessão do Google Earth Engine inicializada.")


## 2. Conferindo o modelo cliente-servidor

Observe a diferença entre imprimir o objeto `ee.Number` (receita no servidor) e pedir o resultado com `.getInfo()`.


In [ ]:
soma_servidor = ee.Number(2).add(3)
print("Objeto no servidor (sem executar):", soma_servidor)
print("Valor trazido para o cliente com getInfo():", soma_servidor.getInfo())

dem = ee.Image("USGS/SRTMGL1_003")
print("Bandas do SRTM:", dem.bandNames().getInfo())


## 3. Região de estudo e busca no catálogo Sentinel-2

Vamos definir uma caixa ao redor de Cuiabá (MT) e buscar cenas da coleção harmonizada `COPERNICUS/S2_SR_HARMONIZED` durante parte da estação seca de 2024. Lembre-se de que a data final em `filterDate` é exclusiva e de que `size()` conta cenas (tiles MGRS), não necessariamente datas distintas, pois a região cruza a borda entre os tiles `21LWC` e `21LXC`.


In [ ]:
ponto_cuiaba = ee.Geometry.Point([-56.0967, -15.6014])
regiao = ponto_cuiaba.buffer(15000).bounds()

colecao_bruta = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(regiao)
    .filterDate("2024-07-01", "2024-10-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

print("Quantidade de cenas (tiles) filtradas:", colecao_bruta.size().getInfo())
print(
    "Tiles MGRS presentes:",
    colecao_bruta.aggregate_array("MGRS_TILE").distinct().getInfo(),
)


## 4. Fator de escala e máscara de nuvens

A função abaixo executa inteiramente no servidor via `.map()`:

- remove pixels marcados na banda `SCL` como saturado/defeituoso (1), sombra de nuvem (3), nuvem média (8), nuvem alta (9), cirros (10) ou neve/gelo (11);
- converte as bandas ópticas selecionadas para reflectância de superfície multiplicando por `0.0001`;
- preserva o metadado `system:time_start` com `copyProperties`.

Como alternativa para comparar visualmente, há uma versão comentada usando `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED`.


In [ ]:
BANDAS_OPTICAS = ["B2", "B3", "B4", "B8", "B11", "B12"]


def preparar_s2_scl(imagem):
    scl = imagem.select("SCL")
    mascara = (
        scl.neq(1)
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    opticas = imagem.select(BANDAS_OPTICAS).multiply(0.0001)
    return opticas.updateMask(mascara).copyProperties(imagem, ["system:time_start"])


colecao_limpa = colecao_bruta.map(preparar_s2_scl)
composta = colecao_limpa.median()

# Alternativa com Cloud Score+ (descomente para comparar visualmente):
# cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
# def preparar_s2_csplus(imagem):
#     mascara = imagem.select("cs_cdf").gte(0.60)
#     opticas = imagem.select(BANDAS_OPTICAS).multiply(0.0001)
#     return opticas.updateMask(mascara).copyProperties(imagem, ["system:time_start"])
# composta_csplus = colecao_bruta.linkCollection(cs_plus, ["cs_cdf"]).map(preparar_s2_csplus).median()


## 5. Índices espectrais (NDVI e NDWI de McFeeters)

Calculamos os índices sobre a imagem de reflectância (`composta`), nunca sobre uma camada `visualize()`. O NDWI abaixo usa verde e NIR (`B3` e `B8`, definição de McFeeters para água superficial).


In [ ]:
ndvi = composta.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndwi = composta.normalizedDifference(["B3", "B8"]).rename("NDWI")

composta_indices = composta.addBands([ndvi, ndwi])
print("Bandas disponíveis na composta:", composta_indices.bandNames().getInfo())


## 6. Visualização interativa com `geemap`

O `.clip(regiao)` é aplicado apenas na hora de exibir. Use o controle de camadas no canto superior direito do mapa para ligar e desligar RGB, falsa cor (NIR-Red-Green), NDVI e NDWI.


In [ ]:
mapa = geemap.Map()
mapa.centerObject(regiao, 11)

vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.30, "gamma": 1.2}
vis_falsa_cor = {"bands": ["B8", "B4", "B3"], "min": 0.02, "max": 0.45, "gamma": 1.1}
vis_ndvi = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["#8c510a", "#d8b365", "#f6e8c3", "#c7eae5", "#5ab4ac", "#01665e"],
}
vis_ndwi = {
    "min": -0.6,
    "max": 0.4,
    "palette": ["#a6611a", "#f5f5f5", "#80cdc1", "#018571"],
}

mapa.addLayer(composta_indices.clip(regiao), vis_rgb, "Sentinel-2 RGB (cor verdadeira)")
mapa.addLayer(composta_indices.clip(regiao), vis_falsa_cor, "Sentinel-2 falsa cor (NIR-R-G)", False)
mapa.addLayer(composta_indices.select("NDVI").clip(regiao), vis_ndvi, "NDVI", False)
mapa.addLayer(composta_indices.select("NDWI").clip(regiao), vis_ndwi, "NDWI (McFeeters)", False)
mapa


## 7. Experimentos sugeridos

1. Troque a janela de datas para a estação chuvosa (`"2024-01-01"` a `"2024-04-01"`) e compare quantas cenas passam pelo filtro de nuvens e como muda a aparência da vegetação.
2. Descomente a versão com `Cloud Score+`, adicione `composta_csplus` ao mapa e procure diferenças em bordas de nuvens ou telhados urbanos muito claros.
3. Altere `min` e `max` em `vis_rgb` e verifique se o valor médio do NDVI mudou (use a aula seguinte ou `reduceRegion` para confirmar que só a tela mudou).
